# 🌿 Agricultural Crop Disease Detection — Evaluation Notebook

**CS 719 · Data Science Project · University of Regina · Winter 2026**  
**Student:** Viral Prajapati (200499893) · **Instructor:** Howard J. Hamilton

---

## Purpose

This notebook loads the **saved DenseNet121 model weights** produced by `training_agricultural.ipynb` and runs a full evaluation on the held-out test set **without re-training**.

### What this notebook produces

| Output | Description |
|--------|-------------|
| Classification Report | Per-class precision, recall, F1-score for all 38 classes |
| Confusion Matrix | 38×38 heatmap visualizing prediction vs. ground truth |
| ROC Curves | One-vs-rest ROC curves with AUC scores for all 38 classes |
| Top-5 / Bottom-5 Classes | Best and worst performing classes by F1-score |
| Sample Predictions | Visual grid showing correct and incorrect predictions |

---

## ⚠️ Before Running

1. Upload the following files (produced by `training_agricultural.ipynb`) to the Colab session or `/content/`:
   - `best_model.weights.h5`
   - `best_model_config.json`
2. Attach the **PlantVillage dataset** (`abdallahalidev/plantvillage-dataset`) in Data settings.
3. Enable **GPU accelerator**: Runtime → Change runtime type → T4 GPU.

## ✅ Step 0 — Check GPU & Environment

In [ ]:
import tensorflow as tf
print("TensorFlow version :", tf.__version__)
print("GPU devices        :", tf.config.list_physical_devices('GPU'))

if not tf.config.list_physical_devices('GPU'):
    print("⚠️  WARNING: No GPU detected. Inference will be slower on CPU.")
else:
    print("✅ GPU available.")

## 📦 Step 1 — Imports

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

import tensorflow as tf
from tensorflow.keras.applications.densenet import preprocess_input as dns_pre
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize
from tensorflow.keras.utils import to_categorical

print("✅ All imports successful")

## ⚙️ Step 2 — Configuration

Must match the values used in `training_agricultural.ipynb`.

In [ ]:
CFG = {
    "img_size"     : 224,
    "batch_size"   : 32,
    "num_classes"  : 38,
    "random_state" : 42,
    "train_ratio"  : 0.70,
    "val_ratio"    : 0.15,
    "test_ratio"   : 0.15,
    "dataset_path" : None,   # filled below
    "results_dir"  : "/content/eval_results",
    # Paths to saved model files (upload these before running)
    "weights_path" : "/content/best_model.weights.h5",
    "config_path"  : "/content/best_model_config.json",
}

os.makedirs(CFG["results_dir"], exist_ok=True)
print("✅ Configuration set")

## 🌱 Step 3 — Locate PlantVillage Dataset

In [ ]:
INPUT_BASE   = "/content"
DATASET_PATH = None

# Search for the 'color' subfolder
for root, dirs, files in os.walk(INPUT_BASE):
    if os.path.basename(root).lower() == "color" and len(dirs) > 10:
        DATASET_PATH = root
        break

# Fallback paths for Colab / Kaggle
if DATASET_PATH is None:
    for candidate in [
        "/content/plantvillage-dataset/color",
        "/kaggle/input/plantvillage-dataset/color",
        "/kaggle/input/plantvillage-dataset/PlantVillage",
    ]:
        if os.path.isdir(candidate):
            DATASET_PATH = candidate
            break

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Could not find PlantVillage 'color' folder.\n"
        "Attach 'abdallahalidev/plantvillage-dataset' in the Data settings."
    )

CFG["dataset_path"] = DATASET_PATH
print(f"✅ Dataset found: {DATASET_PATH}")

labels = sorted([
    d for d in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, d))
])
print(f"Total disease classes: {len(labels)}")

## 📊 Step 4 — Load Image Paths & Reproduce Test Split

Uses the **same random seed and split ratios** as `training_agricultural.ipynb` to ensure identical test sets.

In [ ]:
all_paths, all_labels = [], []

for label in labels:
    label_dir = os.path.join(DATASET_PATH, label)
    for fname in os.listdir(label_dir):
        if os.path.isfile(os.path.join(label_dir, fname)):
            all_paths.append(os.path.join(label_dir, fname))
            all_labels.append(label)

print(f"Total images: {len(all_paths):,}")

# Reproduce the exact same split used during training
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels,
    test_size=1 - CFG["train_ratio"],
    random_state=CFG["random_state"],
    stratify=all_labels,
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    test_size=CFG["test_ratio"] / (CFG["val_ratio"] + CFG["test_ratio"]),
    random_state=CFG["random_state"],
    stratify=temp_labels,
)

print(f"Test set size : {len(test_paths):,} images ({100*len(test_paths)/len(all_paths):.1f}%)")
print(f"Classes       : {len(set(test_labels))}")

## 🏷️ Step 5 — Label Encoding

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(sorted(set(all_labels)))
CLASS_NAMES = list(label_encoder.classes_)
NUM_CLASSES  = len(CLASS_NAMES)

test_encoded  = label_encoder.transform(test_labels)
test_cat      = to_categorical(test_encoded, num_classes=NUM_CLASSES)

print(f"Number of classes : {NUM_CLASSES}")
print("✅ Labels encoded")

## ⚡ Step 6 — Build tf.data Test Pipeline

In [ ]:
def make_test_dataset(paths, labels_cat, img_size, batch_size):
    """DenseNet121 preprocessing — no augmentation, no shuffle."""
    def load_and_preprocess(path, label):
        img  = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
        img  = tf.cast(tf.image.resize(img, [img_size, img_size]), tf.float32)
        img  = dns_pre(img)   # tf.keras.applications.densenet.preprocess_input
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels_cat))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_ds = make_test_dataset(
    test_paths, test_cat,
    CFG["img_size"], CFG["batch_size"]
)

print("✅ Test dataset pipeline ready")

## 🔄 Step 7 — Load Saved DenseNet121 Model

Loads `best_model_config.json` and `best_model.weights.h5` using `tf-keras` with the `batch_shape` compatibility patch required for TF 2.x.

In [ ]:
# Install tf-keras if not already available
try:
    import tf_keras
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "tf-keras", "-q"], check=True)
    import tf_keras

# Load JSON config
with open(CFG["config_path"], "r") as f:
    config = json.load(f)

# Patch InputLayer: rename batch_shape → batch_input_shape for TF 2.x compatibility
def patch_config(obj):
    if isinstance(obj, dict):
        if (obj.get("class_name") == "InputLayer"
                and "batch_shape" in obj.get("config", {})):
            obj["config"]["batch_input_shape"] = obj["config"].pop("batch_shape")
        for v in obj.values():
            patch_config(v)
    elif isinstance(obj, list):
        for item in obj:
            patch_config(item)

patch_config(config)

model = tf_keras.models.model_from_json(json.dumps(config))
model.load_weights(CFG["weights_path"])

print("✅ DenseNet121 model loaded successfully")
print(f"   Total parameters : {model.count_params():,}")
model.summary()

## 📈 Step 8 — Run Inference on Test Set

In [ ]:
import time

print("Running inference on test set...")
t0 = time.time()

y_pred_prob = model.predict(test_ds, verbose=1)          # shape: (N, 38)
y_pred      = np.argmax(y_pred_prob, axis=1)             # predicted class indices
y_true      = np.concatenate(                            # true class indices
    [np.argmax(y, axis=1) for _, y in test_ds]
)

elapsed = time.time() - t0
test_acc = np.mean(y_pred == y_true) * 100

print(f"\n✅ Inference complete in {elapsed:.1f} s")
print(f"   Test set size  : {len(y_true):,} images")
print(f"   Test Accuracy  : {test_acc:.2f}%")

## 📋 Step 9 — Classification Report (Per-Class Metrics)

In [ ]:
report_dict = classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
)
report_str = classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
)

macro_precision = report_dict["macro avg"]["precision"] * 100
macro_recall    = report_dict["macro avg"]["recall"]    * 100
macro_f1        = report_dict["macro avg"]["f1-score"]  * 100

print("=" * 70)
print(f"  DenseNet121 — Final Test Set Results")
print(f"  Test Accuracy    : {test_acc:.2f}%")
print(f"  Macro Precision  : {macro_precision:.2f}%")
print(f"  Macro Recall     : {macro_recall:.2f}%")
print(f"  Macro F1-Score   : {macro_f1:.2f}%")
print("=" * 70)
print()
print(report_str)

# Save report to CSV
df_report = pd.DataFrame(report_dict).transpose()
df_report.to_csv(os.path.join(CFG["results_dir"], "classification_report.csv"))
print("✅ Classification report saved → eval_results/classification_report.csv")

## 🔢 Step 10 — Confusion Matrix (38×38 Heatmap)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

# Normalize to percentages per row (recall perspective)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(
    cm_norm,
    ax=ax,
    cmap="YlOrRd",
    xticklabels=[c.split("___")[-1].replace("_", " ") for c in CLASS_NAMES],
    yticklabels=[c.split("___")[-1].replace("_", " ") for c in CLASS_NAMES],
    linewidths=0.3,
    linecolor="#e0e0e0",
    annot=False,        # too small to annotate 38x38
    vmin=0, vmax=100,
)
ax.set_title(
    "DenseNet121 — Confusion Matrix (Row-Normalized %)\n"
    f"Test Accuracy: {test_acc:.2f}%  |  Macro F1: {macro_f1:.2f}%",
    fontsize=13, fontweight="bold", pad=14
)
ax.set_xlabel("Predicted Class", fontsize=10)
ax.set_ylabel("True Class", fontsize=10)
ax.tick_params(axis="x", labelsize=6, rotation=90)
ax.tick_params(axis="y", labelsize=6)

plt.tight_layout()
cm_path = os.path.join(CFG["results_dir"], "confusion_matrix.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Confusion matrix saved → eval_results/confusion_matrix.png")

## 📉 Step 11 — ROC Curves (One-vs-Rest, All 38 Classes)

In [ ]:
# Binarize true labels for one-vs-rest ROC
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))  # (N, 38)

# Compute ROC / AUC for every class
fpr_dict, tpr_dict, auc_dict = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr_dict[i], tpr_dict[i], _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

mean_auc = np.mean(list(auc_dict.values()))

# ── Plot all 38 ROC curves on one figure ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 9))

cmap = plt.cm.get_cmap("tab20", NUM_CLASSES)
for i in range(NUM_CLASSES):
    short_name = CLASS_NAMES[i].split("___")[-1].replace("_", " ")
    ax.plot(
        fpr_dict[i], tpr_dict[i],
        color=cmap(i), lw=0.9, alpha=0.65,
        label=f"{short_name} (AUC={auc_dict[i]:.2f})"
    )

ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random classifier")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title(
    f"DenseNet121 — ROC Curves (One-vs-Rest, all 38 classes)\nMean AUC = {mean_auc:.4f}",
    fontsize=12, fontweight="bold"
)
ax.legend(loc="lower right", fontsize=5, ncol=2)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
roc_path = os.path.join(CFG["results_dir"], "roc_curves.png")
plt.savefig(roc_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ ROC curves saved → eval_results/roc_curves.png")
print(f"   Mean AUC across all 38 classes: {mean_auc:.4f}")

## 🏆 Step 12 — Top-5 and Bottom-5 Classes by F1-Score

In [ ]:
# Extract per-class F1 scores
class_f1 = {
    cls: report_dict[cls]["f1-score"]
    for cls in CLASS_NAMES
    if cls in report_dict
}

df_f1 = pd.DataFrame(
    [{"Class": k.split("___")[-1].replace("_", " "), "F1-Score": v}
     for k, v in class_f1.items()]
).sort_values("F1-Score", ascending=False).reset_index(drop=True)

top5    = df_f1.head(5)
bottom5 = df_f1.tail(5)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle(
    "DenseNet121 — Per-Class F1-Score: Top 5 and Bottom 5 Classes",
    fontsize=12, fontweight="bold"
)

axes[0].barh(top5["Class"], top5["F1-Score"], color="#4a7c59", alpha=0.85)
axes[0].set_title("Top 5 Classes", fontweight="bold")
axes[0].set_xlim(0, 1.0)
axes[0].invert_yaxis()
axes[0].spines[["top", "right"]].set_visible(False)
for i, v in enumerate(top5["F1-Score"]):
    axes[0].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

axes[1].barh(bottom5["Class"], bottom5["F1-Score"], color="#c0392b", alpha=0.85)
axes[1].set_title("Bottom 5 Classes (Hardest to Classify)", fontweight="bold")
axes[1].set_xlim(0, 1.0)
axes[1].invert_yaxis()
axes[1].spines[["top", "right"]].set_visible(False)
for i, v in enumerate(bottom5["F1-Score"]):
    axes[1].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

plt.tight_layout()
f1_path = os.path.join(CFG["results_dir"], "top_bottom_f1.png")
plt.savefig(f1_path, dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 5 classes:")
print(top5.to_string(index=False))
print("\nBottom 5 classes:")
print(bottom5.to_string(index=False))

## 🖼️ Step 13 — Sample Predictions Grid

Shows 16 random test images with their true label, predicted label, and confidence score. Correct predictions are framed in green; incorrect in red.

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(test_paths), size=16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(16, 14))
fig.suptitle(
    "DenseNet121 — Sample Predictions (green = correct, red = incorrect)",
    fontsize=13, fontweight="bold"
)

for ax, idx in zip(axes.flatten(), sample_idx):
    # Load and preprocess image
    img_raw  = load_img(test_paths[idx], target_size=(224, 224))
    img_arr  = img_to_array(img_raw)
    img_pre  = dns_pre(img_arr.copy())
    pred_prob = model.predict(np.expand_dims(img_pre, axis=0), verbose=0)[0]
    pred_cls  = int(np.argmax(pred_prob))
    confidence = pred_prob[pred_cls] * 100

    true_cls  = int(y_true[idx])
    correct   = pred_cls == true_cls

    true_name = CLASS_NAMES[true_cls].split("___")[-1].replace("_", " ")
    pred_name = CLASS_NAMES[pred_cls].split("___")[-1].replace("_", " ")

    ax.imshow(img_raw)
    border_color = "#27ae60" if correct else "#c0392b"
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
    ax.set_title(
        f"True : {true_name}\nPred : {pred_name} ({confidence:.1f}%)",
        fontsize=6.5,
        color="#27ae60" if correct else "#c0392b",
        fontweight="bold",
    )
    ax.axis("off")

plt.tight_layout()
sample_path = os.path.join(CFG["results_dir"], "sample_predictions.png")
plt.savefig(sample_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Sample predictions saved → eval_results/sample_predictions.png")

## 💾 Step 14 — Save Summary Results

In [ ]:
summary = {
    "model"             : "DenseNet121",
    "test_images"       : len(y_true),
    "num_classes"       : NUM_CLASSES,
    "test_accuracy_pct" : round(test_acc, 2),
    "macro_precision_pct": round(macro_precision, 2),
    "macro_recall_pct"  : round(macro_recall, 2),
    "macro_f1_pct"      : round(macro_f1, 2),
    "mean_auc"          : round(mean_auc, 4),
    "outputs": [
        "eval_results/classification_report.csv",
        "eval_results/confusion_matrix.png",
        "eval_results/roc_curves.png",
        "eval_results/top_bottom_f1.png",
        "eval_results/sample_predictions.png",
    ]
}

summary_path = os.path.join(CFG["results_dir"], "evaluation_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 60)
print("  EVALUATION COMPLETE")
print("=" * 60)
for k, v in summary.items():
    if k != "outputs":
        print(f"  {k:<25} : {v}")
print("\n  Output files:")
for f in summary["outputs"]:
    print(f"    → {f}")
print("=" * 60)

# Zip everything for easy download
import shutil
shutil.make_archive("/content/eval_results", "zip", CFG["results_dir"])
print("\n✅ eval_results.zip created — download from the Files panel.")